# 00. 评估方法总览

Agent 评估不是「跑一个 benchmark 看分数」就结束了。真正的评估体系分三层：

| 层次 | 方法 | 示例 |
|------|------|------|
| Unit | 单元测试 | 工具调用参数是否正确？ |
| Component | 组件测试 | Agent 在一次交互中是否选了正确的工具？ |
| End-to-end | 端到端测试 | 完整任务的成功率和效率 |

> **检查点**：能区分 outcome evaluation（结果对不对）和 process evaluation（过程好不好）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 三层评估体系
from dataclasses import dataclass

@dataclass
class EvalResult:
    name: str; passed: bool; score: float; detail: str

# Level 1: Unit — 工具调用参数校验
def eval_unit() -> EvalResult:
    tests = [("2+3*4",14),("10/2",5),("sqrt(9)",3)]
    import math
    passed = 0
    for expr, expected in tests:
        try:
            result = eval(expr,{"__builtins__":{},"sqrt":math.sqrt}, {})
            if abs(result-expected) < 0.01: passed += 1
        except: pass
    return EvalResult("unit",passed==len(tests),passed/len(tests),f"{passed}/{len(tests)}")

# Level 2: Component — Agent 决策正确性
def eval_component() -> EvalResult:
    # 模拟 Agent 面对不同任务时选择的工具
    scenarios = [
        ("calculate 2+2", "calculator", "calculator"),
        ("search for python", "search", "search"),
        ("what is weather", "weather", "weather"),
    ]
    correct = sum(1 for task, expected, actual in scenarios if expected == actual)
    return EvalResult("component",correct==len(scenarios),correct/len(scenarios),f"{correct}/{len(scenarios)}")

# Level 3: End-to-end — 轨迹评估
def eval_trajectory() -> EvalResult:
    trace = [
        {"step":1,"action":"search","valid":True,"efficient":True},
        {"step":2,"action":"search","valid":True,"efficient":False},  # redundant
        {"step":3,"action":"done","valid":True,"efficient":True},
    ]
    valid = sum(t["valid"] for t in trace)/len(trace)
    efficient = sum(t["efficient"] for t in trace)/len(trace)
    score = valid*0.4 + efficient*0.6
    return EvalResult("trajectory",score>0.7,score,f"valid={valid:.0%}, efficient={efficient:.0%}")

for fn in [eval_unit, eval_component, eval_trajectory]:
    r = fn()
    print(f"[{'✓' if r.passed else '✗'}] {r.name}: {r.detail} (score={r.score:.2f})")

[✓] unit: 3/3 (score=1.00)
[✓] component: 3/3 (score=1.00)
[✓] trajectory: valid=100%, efficient=67% (score=0.80)


## 好评估的特征

1. **可重复**：同一输入总得同一分数
2. **细粒度**：不只说「好/坏」，能指出哪里好/坏
3. **与业务对齐**：评估指标反映真实用户关心的东西

> **检查点**：能为你的 Agent 设计至少 2 个 unit test、1 个 component test、1 个 E2E test。

## 真实 API 实验

真实 LLM-as-Judge 评估器（`evaluator_demo.py --eval-type llm-judge --use-api`）。


In [3]:
SCRIPT = ROOT / 'agents' / 'evaluation' / 'scripts' / 'evaluator_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--eval-type', 'llm-judge', '--use-api'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


[✓ PASS] llm-judge: correct>wrong as expected: 30 vs 10 (score: 1.00)



## 练习

1. 为 `react_loop_demo.py` 设计一个 E2E 测试：给定 goal="learn about python"，断言 3 步内返回有效答案。
2. 实现一个 `score_trajectory(trace, gold)` 函数：对比 Agent 实际 trace 与理想 trace，计算编辑距离。
3. 思考：Agent 的「效率」如何量化？步数最少 = 最好？有没有可能步数多但更安全的情况？